# CareerLens AI — Phase 4: Data Preprocessing & Feature Engineering

In this phase, we clean, normalize, aggregate, and structure our occupational and skill datasets into a unified format for Machine Learning recommendation.

### Preprocessing Objectives:
1. **Clean Text**: Normalize whitespace and preserve essential programming tokens (C++, C#, .NET).
2. **Aggregate Software Skills**: Group technical software tools per occupation.
3. **Aggregate Transferable Skills**: Filter and attach foundational human competencies with high importance (Importance $\ge 3.0$).
4. **Feature Engineering (`Combined_Features`)**: Build a composite lexical representation per occupation combining title, description, and skills for TF-IDF vectorization.
5. **Extract Skills Catalog**: Generate a master vocabulary list of all 8,778 unique skills for autocomplete and skill-gap matching.

## 1. Imports and Setup

In [ ]:
import os
import re
import pandas as pd

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
PROCESSED_PATH = os.path.join(DATA_DIR, "processed_occupations.csv")
SKILLS_PATH = os.path.join(DATA_DIR, "unique_skills.csv")

df_occ = pd.read_csv(os.path.join(DATA_DIR, "occupation_data.csv"))
df_soft = pd.read_csv(os.path.join(DATA_DIR, "software_skills.csv"))
df_trans = pd.read_csv(os.path.join(DATA_DIR, "transferable_skills.csv"))
print(f"Loaded {len(df_occ):,} occupations, {len(df_soft):,} software skills, and {len(df_trans):,} transferable skills.")

## 2. Text Cleaning Function
Why do we need this function? Raw text often contains inconsistent line breaks (`\n`), tabs, or multiple spaces that confuse NLP tokenizers.

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    cleaned = re.sub(r'\s+', ' ', text)
    return cleaned.strip()

# Test cleaning
sample = "  Software   Developer \n\n with   Python,  C++, and .NET   "
print(f"Before: '{sample}'")
print(f"After:  '{clean_text(sample)}'")

## 3. Aggregate Software Skills & Transferable Skills per Occupation

In [ ]:
# Software skills aggregation
df_soft["Workplace Example"] = df_soft["Workplace Example"].apply(clean_text)
df_soft_clean = df_soft.dropna(subset=["Workplace Example"]).drop_duplicates(subset=["O*NET-SOC Code", "Workplace Example"])

soft_agg = df_soft_clean.groupby("O*NET-SOC Code")["Workplace Example"].agg(
    Software_Skills=lambda x: ", ".join(sorted(set(x))),
    Software_Skills_Count=lambda x: len(set(x))
).reset_index()

# Transferable skills aggregation (Importance >= 3.0)
df_trans_im = df_trans[(df_trans["Scale ID"] == "IM") & (df_trans["Data Value"] >= 3.0)].copy()
df_trans_im["Element Name"] = df_trans_im["Element Name"].apply(clean_text)

trans_agg = df_trans_im.groupby("O*NET-SOC Code")["Element Name"].agg(
    Transferable_Skills=lambda x: ", ".join(sorted(set(x))),
    Transferable_Skills_Count=lambda x: len(set(x))
).reset_index()

print(f"Aggregated software skills for {len(soft_agg):,} occupations.")
print(f"Aggregated transferable skills for {len(trans_agg):,} occupations.")

## 4. Merge Tables & Construct `Combined_Features`
We construct a composite text string per occupation. We intentionally include the **Title** twice to give it additional lexical weight during TF-IDF vectorization.

In [ ]:
df_merged = pd.merge(df_occ, soft_agg, on="O*NET-SOC Code", how="left")
df_merged = pd.merge(df_merged, trans_agg, on="O*NET-SOC Code", how="left")

df_merged["Software_Skills"] = df_merged["Software_Skills"].fillna("")
df_merged["Software_Skills_Count"] = df_merged["Software_Skills_Count"].fillna(0).astype(int)
df_merged["Transferable_Skills"] = df_merged["Transferable_Skills"].fillna("")
df_merged["Transferable_Skills_Count"] = df_merged["Transferable_Skills_Count"].fillna(0).astype(int)

df_merged["Combined_Features"] = (
    df_merged["Title"] + " " +
    df_merged["Title"] + " " +
    df_merged["Description"] + " " +
    df_merged["Software_Skills"] + " " +
    df_merged["Transferable_Skills"]
).apply(clean_text)

df_merged.head(2)

## 5. Verify the Final Preprocessed Dataset
Let's inspect a sample occupation to verify the rich feature profile.

In [ ]:
dev_sample = df_merged[df_merged["Title"].str.contains("Software Developers", case=False, na=False)].iloc[0]
print(f"Title: {dev_sample['Title']}")
print(f"Software Skills Count: {dev_sample['Software_Skills_Count']}")
print(f"Transferable Skills Count: {dev_sample['Transferable_Skills_Count']}")
print(f"First 200 characters of Combined_Features:\n{dev_sample['Combined_Features'][:200]}...")